In [84]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [85]:
df=pd.read_csv(r"C:\Users\deves\OneDrive\air_quality_dataset_final.csv") 
print("First 5 rows")
print(df.head())
print("\nDataset Info")
print(df.info())

First 5 rows
         CO        NO2  Temperature   Humidity Air_Quality       city  \
0  1.935247  91.309066    19.890579  30.993778        Poor     Mumbai   
1  4.758500  83.776031    26.053681  26.685614        Poor     Mumbai   
2  3.686770  59.145173    13.472542  29.655744        Poor      Delhi   
3  3.033427  79.049772    25.639296  53.144258        Poor  Bangalore   
4  0.864491  61.113529    17.107847  81.917382    Moderate   Dehradun   

      disease past_medical_record  
0        COPD                 Yes  
1  Bronchitis                 Yes  
2        COPD                 Yes  
3         NaN                 Yes  
4         NaN                 Yes  

Dataset Info
<class 'pandas.DataFrame'>
RangeIndex: 102000 entries, 0 to 101999
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   CO                   96887 non-null   float64
 1   NO2                  96897 non-null   float64
 2   Temperatur

In [86]:
print("\nMissing Values")
print(df.isnull().sum())


Missing Values
CO                      5113
NO2                     5103
Temperature             5098
Humidity                5103
Air_Quality                0
city                       0
disease                20301
past_medical_record        0
dtype: int64


In [87]:
# Numerical columns
num_cols=['CO','NO2','Temperature','Humidity']
for col in num_cols:
    df[col]=df[col].fillna(df[col].median())
# Categorical columns
cat_cols=['city','disease','past_medical_record']
for col in cat_cols:
    df[col]=df[col].fillna(df[col].mode()[0])

In [88]:
for col in num_cols:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3-Q1
    lower=Q1-1.5*IQR
    upper=Q3+1.5*IQR
    df[col]=np.clip(df[col],lower,upper)

In [89]:
le=LabelEncoder()
df['city']=le.fit_transform(df['city'])
df['disease']=le.fit_transform(df['disease'])
df['past_medical_record']=le.fit_transform(df['past_medical_record'])
df['Air_Quality']=le.fit_transform(df['Air_Quality'])

In [90]:
X=df.drop('Air_Quality',axis=1)
y=df['Air_Quality']

In [91]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [92]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [93]:
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train,y_train)
y_pred_knn=knn.predict(X_test)

In [94]:
lr=LogisticRegression(max_iter=200)
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)

In [95]:
def evaluate_model(y_test,y_pred,model_name):
    print(f"\n{model_name}Performance")
    print("Accuracy:",accuracy_score(y_test,y_pred))
    print("Precision:",precision_score(y_test, y_pred,average='weighted'))
    print("Recall:",recall_score(y_test,y_pred,average='weighted'))
    print("F1 Score:",f1_score(y_test,y_pred,average='weighted'))
    cm = confusion_matrix(y_test,y_pred)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
    plt.title(model_name+"Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()